## Read Me

This is a flexible notebook for stepping through an annotation step by step given a specific scenario input. The notebook must be modified in order to select the directory and input file. This is not ideal, so we will have to come up with a better system!

## Set up

In [1]:
import json # !pip
import os
import textwrap
import pandas as pd
import importlib
import numpy as np
from pathlib import Path
import sys

In [2]:
ROOT_DIR = os.getcwd() + '/../'
sys.path.append(ROOT_DIR)

sys.path.append(ROOT_DIR+'src/')
print(ROOT_DIR)

/Users/anna/Dropbox/2023_AOI/MoralLearning/CodeSets/graph_extract/run_annotation/../


In [3]:
import src.annotate_scenario as annotate_scenario
import src.prompts as prompts
import src.translate_to_vis as translate_to_vis
import src.node as node
import src.get_emb_distances as get_emb_distances
import src.utils as utils
importlib.reload(annotate_scenario)
importlib.reload(translate_to_vis)

<module 'src.translate_to_vis' from '/Users/anna/Dropbox/2023_AOI/MoralLearning/CodeSets/graph_extract/run_annotation/../src/translate_to_vis.py'>

In [4]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

## Selet Scenario Input File

In [5]:
# set main paths
SCENARIO_DIR = ROOT_DIR + "scenarios_inputs/" + "cheung_variants/"
# DATA_DIR_HUMAN = ROOT_DIR + "human_data/" 
OUTPUT_DIR = ROOT_DIR + "annotated_outputs/" + "cheung_variants/"

In [69]:
#set scenario file filename
FILENAME = 'rope_ladder.json'

#select scenario and action choice
SCENARIO_ID = 3
ACT_ID = '1'

#read in the scenario
scenario_json = utils.open_scenario(SCENARIO_DIR, FILENAME, SCENARIO_ID, ACT_ID)


Scenario Text: 


It is 1987 and you are on a ferry from Belgium to England. Suddenly, the ferry starts tilting and
water begins to pour in. You and many other passengers are trying to get to the deck by a rope
ladder. You climb up the ladder and stand on the deck, looking at the 20 other passengers making
their way up behind you. Directly below you, a man who is midway up the ladder seems frozen into
immobility by fear or cold and is blocking the way. You try to speak and reach to him, but he does
not react. People behind you are jostling. The ship seems to be sinking fast and the man is still
blocking the ladder. From the crowd below, someone shouts that you should push the man off. If you
push the man off the ladder, he will probably die, but the 19 other people will be able to climb on
deck. If you do not push the man off the ladder, he will probably continue blocking the way so that
many of the people behind you will not be able to get on deck and therefore will drown. 




In [71]:
# print scenario json
print(json.dumps(scenario_json, indent=4))

{
    "id": 3,
    "scenario_title": "Rope Ladder",
    "deontology_level": "3",
    "utility_level": "3",
    "text": "It is 1987 and you are on a ferry from Belgium to England. Suddenly, the ferry starts tilting and water begins to pour in. You and many other passengers are trying to get to the deck by a rope ladder. You climb up the ladder and stand on the deck, looking at the 20 other passengers making their way up behind you. Directly below you, a man who is midway up the ladder seems frozen into immobility by fear or cold and is blocking the way. You try to speak and reach to him, but he does not react. People behind you are jostling. The ship seems to be sinking fast and the man is still blocking the ladder. From the crowd below, someone shouts that you should push the man off. If you push the man off the ladder, he will probably die, but the 19 other people will be able to climb on deck. If you do not push the man off the ladder, he will probably continue blocking the way so th

#### There are 4 major stages of processing.

0. Entities
Label the entities (no human data)

1. Value Scores / "Deontology"
Score the action in moral value

2. Outcomes
Map action to probable outcomes 

3. Outcome Utilities
Consequentialist analysis of harms/benefits of each outcome to each entity

4. Outcome Links
Connection between each entity and each outcome, in terms of Cause, Intend, and Desire



## Go through annotation process step by step (replicates main function in annotate_scenario) 

In [72]:

# get the action choice and convert to two pronoun options (I and Ziv)
this_act = scenario_json['options'][ACT_ID]
this_act_I = this_act
this_act_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_act_I)
print('\n\nAction choice:') 
print(this_act_Ziv)
print(this_act_I)

#get the scenario and convert to two pronoun options
this_scenario = scenario_json['text']
print(scenario_json['text'])
this_scenario_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_scenario)
print("\n\nScenario:")
print(this_scenario_Ziv)
print(this_scenario)


# create a dictionary to write out to csv later
scenario_dict = {'scenario': this_scenario, 'scenario_idx': scenario_json['id'],
                    'choice': this_act_I}





Action choice:
Ziv pushes the man.
push the man
It is 1987 and you are on a ferry from Belgium to England. Suddenly, the ferry starts tilting and water begins to pour in. You and many other passengers are trying to get to the deck by a rope ladder. You climb up the ladder and stand on the deck, looking at the 20 other passengers making their way up behind you. Directly below you, a man who is midway up the ladder seems frozen into immobility by fear or cold and is blocking the way. You try to speak and reach to him, but he does not react. People behind you are jostling. The ship seems to be sinking fast and the man is still blocking the ladder. From the crowd below, someone shouts that you should push the man off. If you push the man off the ladder, he will probably die, but the 19 other people will be able to climb on deck. If you do not push the man off the ladder, he will probably continue blocking the way so that many of the people behind you will not be able to get on deck and t

In [73]:

#initialize Graph object    
g = annotate_scenario.node.Graph()
g.reset()   
print('Graph g initialized and reset.')
g.set_version('d6d780129095bb9540a7957befb2014ea42d92c1')



Graph g initialized and reset.


#### Step 0. Get entities

In [74]:
#Step 0. Get entities

# identify all sentient beings, returning both pronoun forms and a string list
returned_beings = annotate_scenario.process_beings(this_scenario,this_act,g)
beings_I = returned_beings[0]
beings_Ziv = returned_beings[1]
beings_str_list = returned_beings[2]

#update the scenario dict with the beings
scenario_dict["entities"] = beings_str_list



Identified these entities: 

I
The man midway up the ladder who is frozen and blocking the way
19 other passengers behind the man on the ladder


#### Step 1. Deontology / Action Value Scores

In [75]:
#Step 1.  #ACTION VALUE SCORES

#call the process_values function to rate the moral goodness or wrongness of the action with no context
processed_values  = annotate_scenario.process_values_simple(this_scenario, this_act_I, this_act,g) 
print(processed_values)

    

According to deontological moral theories, the morality of a choice is evaluated based not on the states of affairs those choices bring about, but on the nature of the action itself. These theories hold that some choices cannot be justified by their effects—that no matter how morally good their consequences, some choices are morally forbidden. Instead, what makes a choice right is its conformity with a moral norm. Consider the following action. Please rate how well this action conforms to deontic moral norms, using a scale from -100 to 100, where -100 indicates that this action is extremely morally wrong and violates moral norms in a severe way, and +100 indicates that this action is extremely morally right and conforms to moral norms in a very strong way. A rating of 0 indicates that the action is morally neutral in terms of conformity with moral norms. Please return a json object with key:value pair of "score": rating. Please rate this action: push the man
{'score': -50}


#### Step 2. Outcome Likelihoods

In [41]:
#Step 2. Outcomes
processed_events = annotate_scenario.process_outcomes(this_scenario, this_act)
events_I= processed_events[1]
events_Ziv= processed_events[0]
print("\n".join(events_I))         
scenario_dict["outcomes"]= events_I

I and my co-conspirators detain the dictator
The dictator is disempowered
The dictator's family and followers lose power
The regime's killings are halted
The country gains a chance for a new start
Potential reduction in political and ethnic violence
Possible political instability during transition
I and my co-conspirators face risks from loyalists of the dictator


#### Step 3. Outcome Utilities

In [42]:
#Step 3. Outcome utilities

impacts_list = annotate_scenario.process_impacts(this_scenario_Ziv, this_act, this_act_Ziv, events_Ziv, events_I,beings_Ziv,g) 



Processing impacts of event: I and my co-conspirators detain the dictator
Scored impacts for these beings:
['Ziv', '6 co-conspirators', 'the dictator', "the dictator's family", "the dictator's followers", 'people in the country']
Scored values:
[50, 50, -100, -80, -70, 70]

Processing impacts of event: The dictator is disempowered
Scored impacts for these beings:
['Ziv', '6 co-conspirators', 'the dictator', "the dictator's family", "the dictator's followers", 'people in the country']
Scored values:
[80, 80, -100, -90, -80, 90]

Processing impacts of event: The dictator's family and followers lose power
Scored impacts for these beings:
['Ziv', '6 co-conspirators', 'the dictator', "the dictator's family", "the dictator's followers", 'people in the country']
Scored values:
[50, 50, -100, -100, -100, 70]

Processing impacts of event: The regime's killings are halted
Scored impacts for these beings:
['Ziv', '6 co-conspirators', 'the dictator', "the dictator's family", "the dictator's follo

#### Step 4. Cause / Intend / Know Links

In [19]:
#Step 4. causal / intentional / knowledge links -- run on currently generated event/outcome list
output_links = annotate_scenario.process_causal_links(this_scenario_Ziv, events_Ziv, events_I, this_act_Ziv,g)    


Processing event: The trolley runs over and kills two people
{'cause': 'no', 'intend': 'no', 'know': 'yes'}
CKI links for I
C-I-K+

Processing event: I do not intervene in the trolley's path.
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: One person on the side track is not harmed
{'cause': 'yes', 'intend': 'no', 'know': 'no'}
CKI links for I
C+I-K-

Processing event: I witness the deaths of two people.
{'cause': 'yes', 'intend': 'no', 'know': 'yes'}
CKI links for I
C+I-K+


#### Step 5. Write out the results

In [ ]:
#optional -- write out the results 

this_output_filename = f"nie_scenarios_{scenario_json["id"]}_choice_{ACT_ID}.json"
print('\n\nWriting to file: '+this_output_filename)
g_print = g.print_graph()
utils.write_jsonlines(this_output_filename, g_print)
print('\n\n')


translate_to_vis.main(this_output_filename)




Writing to file: nie_scenarios_12_choice_2.json



nie_scenarios_12_choice_2.json
